# Notebook 07: Kiểm Thử Toàn Vẹn & Audit Dataset V2 (KLTN_prep_v2)

> **Mục tiêu của Notebook:**
> 1. **Kiểm tra Checksum SHA-256:** Quét và đối chiếu tính toàn vẹn 3 file zip dataset-v2 (`80_10_10`, `70_15_15`, `60_20_20`).
> 2. **Xác minh Phân chia Dữ liệu (Nested Split Verification):** Tính toán chính xác tỷ lệ phân chia train/val/test trên dữ liệu gốc (lọc bỏ ảnh augment offline và tính chuẩn train unaugmented), xác nhận 0% data leakage.
> 3. **Giải nén & Audit Nhãn / Bounding Box:** Kiểm tra tính hợp lệ của toàn bộ 48 class IDs (0..47) và tọa độ bbox chuẩn hóa [0, 1].
> 4. **Tạo Cấu Hình `data_colab_active.yaml`:** Tự động đọc danh sách class từ `data_{SELECTED_RATIO}.yaml` trong thư mục giải nén, thiết lập đường dẫn tuyệt đối chuẩn cho YOLO.
> 
> *Lưu ý: Sau khi audit xong bước 4, vui lòng mở **Notebook 08** để thực hiện huấn luyện thực nghiệm (Experiment Training & Ablation).*

---

## 📌 Bước 1️⃣: Kết Nối Google Drive & Chuẩn Bị Môi Trường
> **Yêu cầu trước khi chạy:**
> - Tạo lối tắt (Shortcut) của thư mục chứa dataset hoặc thư mục chung `KLTN_38` / `KLTN-2026-testingNtraining` vào **My Drive** của bạn.

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

POSSIBLE_DATASET_PATHS = [
    "/content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/dataset-v2",
    "/content/drive/MyDrive/KLTN_38/dataset/dataset-v2",
    "/content/drive/MyDrive/dataset/dataset-v2",
    "/content/drive/MyDrive/dataset-v2"
]

DATASET_V2_DIR = None
for p in POSSIBLE_DATASET_PATHS:
    if os.path.exists(p):
        DATASET_V2_DIR = p
        break

if DATASET_V2_DIR is None:
    my_drive = Path("/content/drive/MyDrive")
    found = list(my_drive.rglob("dataset-v2"))
    if found:
        DATASET_V2_DIR = str(found[0])

if DATASET_V2_DIR is None:
    print("⚠️ Chưa tự động tìm thấy thư mục dataset-v2 theo đường dẫn mặc định.")
    print("👉 Hãy gán thủ công DATASET_V2_DIR nếu bạn đặt tên khác.")
    DATASET_V2_DIR = "/content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/dataset-v2"
else:
    print(f"✅ Đã tìm thấy thư mục Dataset V2 tại: {DATASET_V2_DIR}")

!rm -rf /content/klcn2026
!git clone https://github.com/mizzhau/yolo11n-cbam-mvtec-defect-detection.git /content/klcn2026
%cd /content/klcn2026
!git checkout main
!git pull origin main

!pip install -q ultralytics==8.3.253 albumentations==2.0.8 tabulate pyyaml
import ultralytics
assert ultralytics.__version__ == "8.3.253", f"Phiên bản Ultralytics không khớp: {ultralytics.__version__}"
print("\n🚀 Môi trường đã sẵn sàng!")

## 🔍 Bước 2️⃣: GIAI ĐOẠN 1 — Kiểm Tra Toàn Vẹn SHA256 & Phân Tích Cấu Trúc File ZIP
> Cell này quét 3 file zip (`80_10_10`, `70_15_15`, `60_20_20`) để:
> 1. Tính toán mã băm SHA256 và đối chiếu với file `.sha256`.
> 2. Đọc trực tiếp các file danh sách `train_*.txt`, `val_*.txt`, `test_*.txt` bên trong gói `KLTN_prep_v2` để tính tỉ lệ phân chia thực tế chính xác 100%.

In [ ]:
import hashlib
import zipfile
import os
from pathlib import Path
from tabulate import tabulate

assert DATASET_V2_DIR and os.path.exists(DATASET_V2_DIR), f"❌ Thư mục không tồn tại: {DATASET_V2_DIR}"
BASE = Path(DATASET_V2_DIR)

SPLITS = {"80_10_10": (80, 10, 10), "70_15_15": (70, 15, 15), "60_20_20": (60, 20, 20)}

def find_zip(key):
    hits = list(BASE.rglob(f"KLTN_prep_v2_{key}.zip"))
    return hits[0] if hits else None

def sha256(p, chunk=1<<24):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()

rows = []
found_zips = {}

print("=" * 80)
print("🔍 BẮT ĐẦU KIỂM TRA SHA256 VÀ PHÂN TÍCH TỆP ZIP TRÊN GOOGLE DRIVE")
print("=" * 80)

for key, ratio in SPLITS.items():
    zp = find_zip(key)
    if zp is None:
        rows.append([key, "MISSING", "N/A", "N/A", "❌ Không tìm thấy zip"])
        continue

    found_zips[key] = str(zp)
    shap = zp.with_name(zp.name + ".sha256")
    exp = shap.read_text(encoding="utf-8").split()[0].lower() if shap.exists() else None
    act = sha256(zp).lower()
    sha_st = "⚠️ NO .sha256" if exp is None else ("✅ KHỚP" if act == exp else "❌ LỆCH")

    try:
        with zipfile.ZipFile(zp) as z:
            namelist = [n for n in z.namelist() if not n.endswith("/")]

            def read_lines_from_zip(filename):
                target = [n for n in namelist if n.endswith(filename)]
                if target:
                    content = z.read(target[0]).decode("utf-8", errors="ignore")
                    return [line.strip() for line in content.splitlines() if line.strip() and not line.strip().startswith("#")]
                return []

            train_rall = read_lines_from_zip(f"train_{key}_Rall.txt")
            orig_train = [l for l in train_rall if "/aug/" not in l] if train_rall else []

            val_lines = read_lines_from_zip(f"val_{key}.txt")
            test_lines = read_lines_from_zip(f"test_{key}.txt")
            train_active = read_lines_from_zip(f"train_{key}.txt")

            n_orig_train = len(orig_train) if orig_train else len(train_active)
            n_val = len(val_lines)
            n_test = len(test_lines)
            tot_orig = n_orig_train + n_val + n_test

            if tot_orig > 0:
                actual_ratio = (round(100 * n_orig_train / tot_orig, 1), round(100 * n_val / tot_orig, 1), round(100 * n_test / tot_orig, 1))
                ratio_ok = all(abs(a - e) < 1.0 for a, e in zip(actual_ratio, ratio))
                split_summary = f"Gốc: {n_orig_train}/{n_val}/{n_test} → {actual_ratio}% {'✅ KHỚP' if ratio_ok else '⚠️ LỆCH'} | Active R=1: {len(train_active)}"

                stems = lambda S: {Path(x).stem for x in S}
                overlap = len(stems(orig_train or train_active) & stems(val_lines)) + \
                          len(stems(orig_train or train_active) & stems(test_lines)) + \
                          len(stems(val_lines) & stems(test_lines))
                leakage_str = f"Trùng: {overlap} {'✅ 0%' if overlap == 0 else '❌ LEAK'}"
            else:
                img_count = sum(1 for n in namelist if n.lower().endswith((".jpg", ".png", ".jpeg")))
                split_summary = f"Tổng {img_count} ảnh (Chưa đọc được file split list)"
                leakage_str = "-"

            rows.append([key, zp.name, f"{sha_st} ({act[:8]})", split_summary, leakage_str])
    except Exception as e:
        rows.append([key, zp.name, "CORRUPT", "Error", str(e)])

print(tabulate(rows, headers=["Tỉ lệ", "Zip", "SHA256", "Phân chia Gốc & Train Active", "Leakage Check"], tablefmt="grid"))

## 🧪 Bước 3️⃣: Giải Nén Thử Nghiệm & Audit Chi Tiết Nhãn
> Chọn 1 bộ dataset (mặc định: `70_15_15`) để giải nén lên ổ cứng SSD cục bộ của Colab (`/content/dataset_v2/`).
> Sau đó chạy script kiểm tra nhãn:
> - Tự động bỏ qua các file cấu hình, file mô tả hoặc dòng comment `#`.
> - Kiểm tra Class ID có hợp lệ trong khoảng `0..47`.
> - Kiểm tra tọa độ Bounding Box `[x_center, y_center, width, height]` có chuẩn hóa đúng `[0, 1]`.

In [ ]:
import os
import yaml
from glob import glob
from pathlib import Path

SELECTED_RATIO = "70_15_15"  # Tùy chọn: '80_10_10', '70_15_15', '60_20_20'

TARGET_ZIP = found_zips.get(SELECTED_RATIO) if 'found_zips' in locals() and SELECTED_RATIO in found_zips else None
if TARGET_ZIP is None:
    hits = list(Path(DATASET_V2_DIR).rglob(f"KLTN_prep_v2_{SELECTED_RATIO}.zip"))
    assert hits, f"❌ Không tìm thấy zip cho {SELECTED_RATIO}"
    TARGET_ZIP = str(hits[0])

EXTRACT_DIR = f"/content/dataset_v2_{SELECTED_RATIO}"

print(f"📦 Đang giải nén {TARGET_ZIP} vào {EXTRACT_DIR}...")
!mkdir -p "{EXTRACT_DIR}"
!unzip -q -o "{TARGET_ZIP}" -d "{EXTRACT_DIR}"
print(f"✅ Giải nén hoàn tất vào {EXTRACT_DIR}!")

print("\n📂 Cấu trúc thư mục mức 1 và 2:")
!ls -la "{EXTRACT_DIR}"
if os.path.exists(f"{EXTRACT_DIR}/KLTN_prep_v2"):
    !ls -la "{EXTRACT_DIR}/KLTN_prep_v2"

# Tìm đúng các file nhãn YOLO trong thư mục labels/
all_label_files = glob(f"{EXTRACT_DIR}/**/labels/**/*.txt", recursive=True)
if not all_label_files:
    # Fallback nếu labels đặt phẳng
    candidates = glob(f"{EXTRACT_DIR}/**/*.txt", recursive=True)
    all_label_files = [f for f in candidates if not any(x in f for x in ["train_", "val_", "test_", "requirements", "config", "classes", ".sha256"])]

print(f"\n🔍 Tìm thấy tổng cộng {len(all_label_files)} file nhãn khuyết tật (.txt).")

invalid_classes = set()
invalid_bboxes = 0
total_objects = 0

for lf in all_label_files:
    with open(lf, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            clean_line = line.strip()
            # Bỏ qua dòng trống hoặc dòng comment
            if not clean_line or clean_line.startswith("#"):
                continue
            parts = clean_line.split()
            if len(parts) >= 5:
                # Thử parse class ID, bỏ qua nếu dòng header/text
                try:
                    cls_id = int(parts[0])
                except ValueError:
                    continue
                total_objects += 1
                if cls_id < 0 or cls_id >= 48:
                    invalid_classes.add(cls_id)
                coords = [float(x) for x in parts[1:5]]
                if any(c < 0.0 or c > 1.0 for c in coords):
                    invalid_bboxes += 1

print(f"📊 Tổng số bounding boxes: {total_objects}")
if invalid_classes:
    print(f"❌ CẢNH BÁO: Phát hiện Class ID nằm ngoài dải 0..47: {invalid_classes}")
else:
    print("✅ Toàn bộ Class ID đều hợp lệ trong khoảng 0..47!")

if invalid_bboxes > 0:
    print(f"⚠️ Có {invalid_bboxes} bbox có tọa độ vượt ngưỡng [0, 1].")
else:
    print("✅ Toàn bộ tọa độ Bounding Box đều chuẩn hóa đúng trong [0, 1]!")

## 🛠️ Bước 4️⃣: Tự Động Tạo / Chuẩn Hóa File `data.yaml` Tuyệt Đối
> Tự động tìm thư mục `KLTN_prep_v2` và kiểm tra file YAML có sẵn (`data_70_15_15.yaml`...) hoặc file `.txt` list để trỏ đường dẫn tuyệt đối cho YOLO huấn luyện.

In [ ]:
import yaml
from pathlib import Path

extract_p = Path(EXTRACT_DIR)
dataset_root = extract_p / "KLTN_prep_v2" if (extract_p / "KLTN_prep_v2").exists() else extract_p

ZIP_DATA_YAML = dataset_root / f"data_{SELECTED_RATIO}.yaml"
assert ZIP_DATA_YAML.exists(), f"❌ Không tìm thấy {ZIP_DATA_YAML}"

with open(ZIP_DATA_YAML, "r", encoding="utf-8") as f:
    zip_cfg = yaml.safe_load(f)

class_names = zip_cfg.get("names", [])
nc = zip_cfg.get("nc", len(class_names))
assert nc == 48 and len(class_names) == 48, f"❌ Số lượng class không khớp 48: nc={nc}, len={len(class_names)}"
print(f"✅ Đã tải và xác thực đúng 48 class names từ: {ZIP_DATA_YAML}")

train_file = dataset_root / f"train_{SELECTED_RATIO}.txt"
val_file = dataset_root / f"val_{SELECTED_RATIO}.txt"
test_file = dataset_root / f"test_{SELECTED_RATIO}.txt"

colab_yaml_content = {
    "path": str(dataset_root),
    "train": str(train_file),
    "val": str(val_file),
    "test": str(test_file),
    "nc": 48,
    "names": class_names
}

ACTIVE_YAML_PATH = "/content/klcn2026/configs/data/exp_2/data_colab_active.yaml"
os.makedirs(os.path.dirname(ACTIVE_YAML_PATH), exist_ok=True)
with open(ACTIVE_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(colab_yaml_content, f, allow_unicode=True, sort_keys=False)

print(f"✅ Đã tạo file YAML cấu hình chuẩn tại: {ACTIVE_YAML_PATH}")
print("\n--- Nội dung YAML ---")
!cat {ACTIVE_YAML_PATH}

## 🎯 Bước 5️⃣: Hoàn Tất Audit & Chuyển Sang Huấn Luyện (Notebook 08)
> **Chúc mừng! Bộ dữ liệu V2 và cấu hình `data_colab_active.yaml` đã được kiểm thử toàn vẹn 100%.**
> 
> 👉 Hãy mở notebook huấn luyện chính thức: **`notebooks/08_Training_v2.ipynb`** để tiến hành chạy thực nghiệm cả 6 mô hình với cấu hình augmentation online `v2` chuẩn hóa.
> 
> *(Cell bên dưới là tùy chọn để bạn chạy thử nghiệm nhanh 1 epoch kiểm tra kết nối GPU trước khi sang Notebook 08)*.

In [ ]:
from src.training.train_cbam import train_cbam

print("🔥 Chạy smoke test 1 epoch (fraction 2%) kiểm tra pipeline GPU...")
train_cbam(
    data_yaml="/content/klcn2026/configs/data/exp_2/data_colab_active.yaml",
    model_cfg="configs/models/yolo11n_cbam_neck_head.yaml",
    epochs=1,
    batch_size=16,
    img_size=640,
    fraction=0.02,
    aug_profile="v2",
    overwrite=True,
    project="/content/smoke_check",
    name="smoke"
)
print("\n✅ Pipeline sẵn sàng 100%! Vui lòng mở Notebook 08 để huấn luyện chính thức.")